In [3]:
import os
import shutil
from pathlib import Path
from uuid import uuid4

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

c:\Users\Hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 1. Set up paths and the vector store

In [4]:
# Resolve the project root so the notebook works from either the repo root or the notebooks folder
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

project_root

WindowsPath('d:/RAG/4_vectorStores')

In [5]:
# Load the enviornment variables from the .env file
dotenv_path = project_root / ".env"
load_dotenv(dotenv_path=dotenv_path)

if not os.getenv("HUGGINGFACEHUB_API_TOKEN"):
    raise ValueError("Please add your API token to the .env file")

print("HuggingFace API token loaded successfully.")

HuggingFace API token loaded successfully.


In [6]:
# Used a fixed collection name and persistance path so each rerun in predictable

collection_name = 'demo'
persist_directory = project_root / 'notebooks' / 'chroma_langchain_db'

print(f"Collection Name: {collection_name}")
print(f"Persist Directory: {persist_directory}")

Collection Name: demo
Persist Directory: d:\RAG\4_vectorStores\notebooks\chroma_langchain_db


In [7]:
# Start fresh so the CRUD Flow produces the same result each time
if persist_directory.exists():
    shutil.rmtree(persist_directory)
    print("Removed the old Chroma Directory")
else:
    print("No previous Chroma Directory was found")

No previous Chroma Directory was found


In [9]:
# Create the embedding model and connect it to a persist Chroma Store.
emembeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vector_store = Chroma(
    collection_name=collection_name,
    embedding_function=emembeddings,
    persist_directory = str(persist_directory)
)

print("Vector Store created successfully.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3052.54it/s]


Vector Store created successfully.


# 2. Add small Helper Functions

In [11]:
def preview_text(text, limit=80):
    """Return a short preview, for clearer notebook output"""
    if len(text) <= limit:
        return text
    return text[:limit]+ "..."

def print_documents(title, docs):
    """Print documents in a readable format"""
    print(title)
    for index, doc in enumerate(docs):
        print(f"{index}. id={doc.id}")
        print(f"    topic={doc.metadata.get('topic')} | doc_number = {doc.metadata.get('doc_number')}")
        print(f"    content={doc.page_content}")
    print()

# 3. Create and Insert Example Documents

In [12]:
# Keep the raw sample data separate from the Document objects so it is easier to read.
document_examples = [
    {
        "topic": "AI",
        "doc_number": 1,
        "text": "Artificial intelligence helps machines perform tasks that usually need human reasoning.",
    },
    {
        "topic": "AI",
        "doc_number": 2,
        "text": "AI systems can analyze patterns in data to support predictions and automation.",
    },
    {
        "topic": "AI",
        "doc_number": 3,
        "text": "Responsible AI development includes fairness, transparency, and safety checks.",
    },
    {
        "topic": "RAG",
        "doc_number": 4,
        "text": "RAG combines retrieval with generation so the model can answer using external knowledge.",
    },
    {
        "topic": "RAG",
        "doc_number": 5,
        "text": "A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.",
    },
    {
        "topic": "RAG",
        "doc_number": 6,
        "text": "Vector stores are important in RAG because they make semantic search over embedded documents possible.",
    },
    {
        "topic": "LLM",
        "doc_number": 7,
        "text": "LLMs generate text by predicting likely next tokens from patterns learned during training.",
    },
    {
        "topic": "LLM",
        "doc_number": 8,
        "text": "Prompt design can improve how clearly an LLM follows instructions and returns useful answers.",
    },
    {
        "topic": "Cricket",
        "doc_number": 9,
        "text": "Cricket teams score runs through batting partnerships, boundaries, and quick running between the wickets.",
    },
    {
        "topic": "Cricket",
        "doc_number": 10,
        "text": "A cricket bowler can pressure batters with pace, swing, spin, and accurate line and length.",
    },
]

print(f"Prepared {len(document_examples)} document examples.")

Prepared 10 document examples.


In [13]:
# Convert the sample data into LangChain Document objects.
documents = [
    Document(
        id=str(uuid4()),
        page_content=item["text"],
        metadata={"topic": item["topic"], "doc_number": item["doc_number"]},
    )
    for item in document_examples
]

print_documents("Dummy documents prepared:", documents)

Dummy documents prepared:
0. id=226b6ce3-583d-48e1-add8-839cfd2d8f7b
    topic=AI | doc_number = 1
    content=Artificial intelligence helps machines perform tasks that usually need human reasoning.
1. id=c803db67-87f8-4d76-b70f-8dc0cbf0b215
    topic=AI | doc_number = 2
    content=AI systems can analyze patterns in data to support predictions and automation.
2. id=6158db9d-5438-410b-ae4f-b5606783e5bf
    topic=AI | doc_number = 3
    content=Responsible AI development includes fairness, transparency, and safety checks.
3. id=e65b5969-184b-45da-b2bc-d92c92b6aced
    topic=RAG | doc_number = 4
    content=RAG combines retrieval with generation so the model can answer using external knowledge.
4. id=aaa98237-2e98-4f13-817b-1fe3bb9594dd
    topic=RAG | doc_number = 5
    content=A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.
5. id=2210b254-06c5-4b1a-80e8-93480bb939b9
    topic=RAG | doc_number = 6
    content=Vector stores are important

In [15]:
# Insert the doccuments into Chroma. Chroma creates embedding furing this step
document_ids = vector_store.add_documents(documents)

print("Inserted document ids:")
for doc_id in document_ids:
    print(doc_id)

print(f"\nTotal inserted documents: {len(document_ids)}")

Inserted document ids:
226b6ce3-583d-48e1-add8-839cfd2d8f7b
c803db67-87f8-4d76-b70f-8dc0cbf0b215
6158db9d-5438-410b-ae4f-b5606783e5bf
e65b5969-184b-45da-b2bc-d92c92b6aced
aaa98237-2e98-4f13-817b-1fe3bb9594dd
2210b254-06c5-4b1a-80e8-93480bb939b9
e304c9b3-b31c-4359-ae25-e80190859b74
f225ac58-f68f-4e2f-b9c9-7972e5886219
270dbbd2-662e-4eb9-99f1-d879f2fc1b9e
df21b537-5f07-476c-a87b-25ba4d2d4974

Total inserted documents: 10


# 4. Read The stored Data Back

In [17]:
# The get() method returns the low-level Chroma record structure.
raw_records = vector_store.get(include=["embeddings", "metadatas", "documents"])
raw_records.keys()

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas'])

# 5. Run A Smiliairty Search

In [18]:
query = "How does a RAG help an LLM answer questions using outside knowledge?"

In [20]:
search_results = vector_store.similarity_search(query, k=3)
print(f"Query: {query}\n")
print_documents("Similarity search results:", search_results)

Query: How does a RAG help an LLM answer questions using outside knowledge?

Similarity search results:
0. id=e65b5969-184b-45da-b2bc-d92c92b6aced
    topic=RAG | doc_number = 4
    content=RAG combines retrieval with generation so the model can answer using external knowledge.
1. id=f225ac58-f68f-4e2f-b9c9-7972e5886219
    topic=LLM | doc_number = 8
    content=Prompt design can improve how clearly an LLM follows instructions and returns useful answers.
2. id=e304c9b3-b31c-4359-ae25-e80190859b74
    topic=LLM | doc_number = 7
    content=LLMs generate text by predicting likely next tokens from patterns learned during training.



# 6. Update Existing Documents

In [21]:
ids_to_update = [document_ids[3], document_ids[7]]

In [22]:
# Keep the replacement text separate so the update step stays easy to follow.
updated_examples = [
    {
        "id": ids_to_update[0],
        "topic": "RAG",
        "doc_number": 4,
        "text": "RAG improves answer quality by retrieving relevant context before the language model generates a response.",
    },
    {
        "id": ids_to_update[1],
        "topic": "LLM",
        "doc_number": 8,
        "text": "Well-written prompts help an LLM stay focused, follow instructions, and produce more reliable outputs.",
    },
]

updated_documents = [
    Document(
        id=item["id"],
        page_content=item["text"],
        metadata={"topic": item["topic"], "doc_number": item["doc_number"]},
    )
    for item in updated_examples
]

print_documents("Updated document content:", updated_documents)

Updated document content:
0. id=e65b5969-184b-45da-b2bc-d92c92b6aced
    topic=RAG | doc_number = 4
    content=RAG improves answer quality by retrieving relevant context before the language model generates a response.
1. id=f225ac58-f68f-4e2f-b9c9-7972e5886219
    topic=LLM | doc_number = 8
    content=Well-written prompts help an LLM stay focused, follow instructions, and produce more reliable outputs.



In [23]:
print([doc.page_content for doc in documents if doc.id in ids_to_update])

['RAG combines retrieval with generation so the model can answer using external knowledge.', 'Prompt design can improve how clearly an LLM follows instructions and returns useful answers.']


In [24]:
vector_store.update_documents(ids=ids_to_update, documents=updated_documents)

print("Updated these ids:")
for doc_id in ids_to_update:
    print(doc_id)

Updated these ids:
e65b5969-184b-45da-b2bc-d92c92b6aced
f225ac58-f68f-4e2f-b9c9-7972e5886219


In [25]:
# Read the updated records back from Chroma to confirm the new values were stored.
updated_raw_records = vector_store.get(ids=ids_to_update)

print("Raw records returned by get(ids=ids_to_update):")
for doc_id, document_text, metadata in zip(
    updated_raw_records["ids"],
    updated_raw_records["documents"],
    updated_raw_records["metadatas"],
):
    print(f"id={doc_id}")
    print(f"metadata={metadata}")
    print(f"content={preview_text(document_text)}")
    print()

Raw records returned by get(ids=ids_to_update):
id=e65b5969-184b-45da-b2bc-d92c92b6aced
metadata={'doc_number': 4, 'topic': 'RAG'}
content=RAG improves answer quality by retrieving relevant context before the language m...

id=f225ac58-f68f-4e2f-b9c9-7972e5886219
metadata={'doc_number': 8, 'topic': 'LLM'}
content=Well-written prompts help an LLM stay focused, follow instructions, and produce ...



In [26]:
updated_query = "How can retrieved context improve an LLM response in RAG?"
updated_query

'How can retrieved context improve an LLM response in RAG?'

In [27]:
updated_search_results = vector_store.similarity_search(updated_query, k=2)
print(f"Updated query: {updated_query}\n")
print_documents("Similarity search after update:", updated_search_results)

Updated query: How can retrieved context improve an LLM response in RAG?

Similarity search after update:
0. id=e65b5969-184b-45da-b2bc-d92c92b6aced
    topic=RAG | doc_number = 4
    content=RAG improves answer quality by retrieving relevant context before the language model generates a response.
1. id=aaa98237-2e98-4f13-817b-1fe3bb9594dd
    topic=RAG | doc_number = 5
    content=A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.



# 7. Delete Documents

In [28]:
# Delete the two cricket examples so the final collection is smaller.
ids_to_delete = [document_ids[8], document_ids[9]]
ids_to_delete

['270dbbd2-662e-4eb9-99f1-d879f2fc1b9e',
 'df21b537-5f07-476c-a87b-25ba4d2d4974']

In [29]:
vector_store.delete(ids=ids_to_delete)

print("Deleted these ids:")
for doc_id in ids_to_delete:
    print(doc_id)

Deleted these ids:
270dbbd2-662e-4eb9-99f1-d879f2fc1b9e
df21b537-5f07-476c-a87b-25ba4d2d4974


In [30]:
remaining_records = vector_store.get()
remaining_ids = remaining_records["ids"]

print(f"Remaining document count: {len(remaining_ids)}")
print("Remaining ids:")
for doc_id in remaining_ids:
    print(doc_id)

print("\nDeleted ids still present?")
for doc_id in ids_to_delete:
    print(f"{doc_id}: {doc_id in remaining_ids}")

Remaining document count: 8
Remaining ids:
226b6ce3-583d-48e1-add8-839cfd2d8f7b
c803db67-87f8-4d76-b70f-8dc0cbf0b215
6158db9d-5438-410b-ae4f-b5606783e5bf
e65b5969-184b-45da-b2bc-d92c92b6aced
aaa98237-2e98-4f13-817b-1fe3bb9594dd
2210b254-06c5-4b1a-80e8-93480bb939b9
e304c9b3-b31c-4359-ae25-e80190859b74
f225ac58-f68f-4e2f-b9c9-7972e5886219

Deleted ids still present?
270dbbd2-662e-4eb9-99f1-d879f2fc1b9e: False
df21b537-5f07-476c-a87b-25ba4d2d4974: False


In [31]:
print([doc.metadata["topic"] for doc in documents if doc.id in remaining_ids])

['AI', 'AI', 'AI', 'RAG', 'RAG', 'RAG', 'LLM', 'LLM']
